# Financial Gravity — is the S&P / Fed balance sheet really "flat since 2008"?
The claim: divide the S&P 500 by the Fed's balance sheet and the line is basically flat since 2008 → the whole
bull market is just liquidity. **We test it instead of betting it.**

### The honest prior (to be confirmed or killed by the data)
- **Spirit likely right:** much of the *nominal* post-2008 bull = balance-sheet/dollar expansion (the debasement /
  "own the numerator" thesis).
- **"Flat" likely OVERSTATED, and breaks after 2022:** the Fed did *quantitative tightening* (QT) from mid-2022 —
  **shrinking** the balance sheet while stocks melted up → the ratio should RISE, not stay flat. That break =
  the AI/liquidity rally that ISN'T the Fed ([[market-fragility]]).
- **Construction-sensitive:** which series (WALCL total assets vs reserves-net-of-RRP/TGA), log vs linear, start
  date — all swing it. **Global M2 may fit the recent rally better than the Fed alone.** We check both.

### The test (not vibes)
1. Plot SPX, balance sheet, and the ratio (log) with QE vs QT eras shaded.
2. **Regress log(ratio) on time** — slope ≈ 0 ⇒ "flat"; trending ⇒ not flat. Do it FULL-sample and QE-only vs QT-only.
3. **How much of log-SPX does log-balance-sheet actually explain?** (R² of SPX~BS). High ⇒ "it's liquidity"; low ⇒ not.


In [ ]:
# 1 · Setup
!pip -q install yfinance >/dev/null 2>&1
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
import yfinance as yf, matplotlib.pyplot as plt
def fred(series):
    '''Pull a FRED series as a dated Series via the public CSV endpoint (no API key).'''
    url=f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}"
    df=pd.read_csv(url)
    df.columns=["date","val"]; df["date"]=pd.to_datetime(df["date"])
    df["val"]=pd.to_numeric(df["val"],errors="coerce")     # FRED uses "." for missing
    return df.dropna().set_index("date")["val"]
print("ready")


In [ ]:
# 2 · CONFIG
START      = "2008-01-01"
SPX_TICKER = "^GSPC"
BS_SERIES  = "WALCL"     # Fed total assets ($MM, weekly). Alt: "WRESBAL" (reserves)
M2_SERIES  = "M2SL"      # US M2 ($B, monthly) — the alternative denominator
QT_START   = "2022-06-01"  # ~ when QT began (for the era split)


In [ ]:
# 3 · Data — SPX + Fed balance sheet + M2, aligned weekly
spx = yf.download(SPX_TICKER, start=START, auto_adjust=True, progress=False)["Close"]
if hasattr(spx,'columns'): spx=spx.iloc[:,0]
bs  = fred(BS_SERIES)/1000.0      # $MM → $B
m2  = fred(M2_SERIES)             # $B
df = pd.concat({"spx":spx,"bs":bs,"m2":m2},axis=1).resample("W").last().ffill().dropna()
df = df[df.index>=START]
print(f"{len(df)} weeks, {df.index[0].date()} → {df.index[-1].date()}")
print(f"SPX {df.spx.iloc[0]:.0f}→{df.spx.iloc[-1]:.0f} ({df.spx.iloc[-1]/df.spx.iloc[0]:.1f}x) | "
      f"Fed BS ${df.bs.iloc[0]/1000:.1f}T→${df.bs.iloc[-1]/1000:.1f}T ({df.bs.iloc[-1]/df.bs.iloc[0]:.1f}x) | "
      f"M2 ${df.m2.iloc[0]/1000:.1f}T→${df.m2.iloc[-1]/1000:.1f}T ({df.m2.iloc[-1]/df.m2.iloc[0]:.1f}x)")
df["ratio_bs"]=df.spx/df.bs; df["ratio_m2"]=df.spx/df.m2


In [ ]:
# 4 · THE TEST — is log(ratio) flat? + how much of SPX does the balance sheet explain?
def slope_r2(y_index):
    y=np.log(y_index.values); x=np.asarray((y_index.index-y_index.index[0]).days,dtype=float)/365.25
    b,a=np.polyfit(x,y,1); yhat=a+b*x
    r2=1-((y-yhat)**2).sum()/((y-y.mean())**2).sum()
    return b, r2   # b = log-growth/yr of the RATIO (0 = flat); r2 = trend fit
def report(name,series):
    b,r2=slope_r2(series)
    print(f" {name:16} ratio trend {np.exp(b)-1:+6.1%}/yr  (0%=flat)  | over {len(series)}w it moved "
          f"{series.iloc[-1]/series.iloc[0]:.2f}x")
print("="*72); print('IS THE RATIO "FLAT"?  (trend near 0%/yr = flat; big = not flat)'); print("="*72)
report("SPX / Fed BS", df["ratio_bs"])
report("SPX / M2",     df["ratio_m2"])
qe=df[df.index<QT_START]; qt=df[df.index>=QT_START]
print("\n-- SPX/Fed-BS by era --")
report("  QE era (pre-6/22)", qe["ratio_bs"])
report("  QT era (6/22→now)", qt["ratio_bs"])

print("\n"+"="*72); print("HOW MUCH OF log-SPX DOES log-BALANCE-SHEET EXPLAIN? (R²; high=it's liquidity)"); print("="*72)
def xr2(y,x):
    y=np.log(y.values); x=np.log(x.values); b,a=np.polyfit(x,y,1)
    return 1-((y-(a+b*x))**2).sum()/((y-y.mean())**2).sum(), b
for lab,den in [("Fed BS","bs"),("M2","m2")]:
    r2,beta=xr2(df["spx"],df[den]); print(f" SPX ~ {lab:7}: R²={r2:.2f}, elasticity={beta:.2f} (full sample)")
for lab,seg in [("QE era",qe),("QT era",qt)]:
    r2,beta=xr2(seg["spx"],seg["bs"]); print(f" SPX ~ Fed BS, {lab}: R²={r2:.2f}, elasticity={beta:.2f}")


In [ ]:
# 5 · SEE it — SPX vs BS (indexed) + the ratio, QT shaded
fig,ax=plt.subplots(2,1,figsize=(13,8),sharex=True,gridspec_kw={"height_ratios":[1,1]})
base=df.index[0]
ax[0].plot(df.index, df.spx/df.spx.iloc[0]*100, label="S&P 500", color="black")
ax[0].plot(df.index, df.bs /df.bs.iloc[0]*100,  label="Fed balance sheet", color="tab:blue")
ax[0].plot(df.index, df.m2 /df.m2.iloc[0]*100,  label="M2", color="tab:green", alpha=.7)
ax[0].set_yscale("log"); ax[0].set_ylabel("indexed=100 @ 2008"); ax[0].legend(fontsize=8)
ax[0].set_title("If SPX & balance sheet move together, the ratio (below) is flat")
ax[1].plot(df.index, df["ratio_bs"]/df["ratio_bs"].iloc[0]*100, color="tab:red", label="SPX / Fed BS")
ax[1].plot(df.index, df["ratio_m2"]/df["ratio_m2"].iloc[0]*100, color="tab:green", alpha=.6, label="SPX / M2")
ax[1].axhline(100,color="k",lw=.5,ls="--")
ax[1].axvspan(pd.to_datetime(QT_START), df.index[-1], color="orange", alpha=.10, label="QT era")
ax[1].set_yscale("log"); ax[1].set_ylabel("ratio, indexed=100 @2008"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Flat = the red line hugs 100. A rising red line in the QT band = the rally the Fed did NOT fund.")


## How to read the verdict
- **"Ratio trend %/yr"** (cell 4): near **0%/yr** ⇒ the claim holds (flat). A big +%/yr ⇒ stocks outran liquidity.
  Watch the **QE-vs-QT split** — the claim likely holds in QE and breaks in QT (ratio rising while the Fed shrank).
- **R² of SPX~BS:** high (say >0.8) ⇒ "the bull market IS the balance sheet." Lower ⇒ liquidity explains *some*
  but leaves a large residual = real earnings / other liquidity. Compare **Fed BS vs M2** — if M2's R² is higher,
  the Fed is the wrong denominator and *broad* money (global liquidity) is the better "gravity."
- **The chart:** flat = the red ratio line hugs 100. If it lifts off in the orange QT band, the post-2022 rally is
  the part liquidity didn't fund — the [[market-fragility]] "is it real" question, isolated.
- **Discipline:** this is one construction. Swapping WALCL→reserves-net-of-RRP/TGA, or Fed→global-M2, or the start
  date, moves it. A single flat-looking chart is a hypothesis, not a proof — that's why we quantified instead of eyeballed.
- If it confirms "liquidity explains most of it": that's the debasement spine ([[new-economy-regime]]) with a number.
  If the QT-era residual is big: the current rally has a non-Fed driver (earnings / AI-capex / global liquidity) that
  the "financial gravity" one-liner misses.
